-- 1. 创建学生选课数据库
CREATE DATABASE IF NOT EXISTS db_experiment_week03
CHARACTER SET utf8mb4
COLLATE utf8mb4_unicode_ci;

-- 2. 使用该数据库
USE db_experiment_week03;

-- 3. 创建 Student 表（学生表）
CREATE TABLE Student (
    ID VARCHAR(20) NOT NULL COMMENT '学号',
    Name VARCHAR(50) NOT NULL COMMENT '姓名',
    Sex ENUM('男', '女') NOT NULL COMMENT '性别',
    Age INT COMMENT '年龄',
    Dept VARCHAR(50) NOT NULL COMMENT '系别',
    PRIMARY KEY (ID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='学生表';

-- 4. 插入学生数据（包含课件中使用的CS系学生）
INSERT INTO Student (ID, Name, Sex, Age, Dept) VALUES
('200215121', '李勇', '男', 20, 'CS'),
('200215122', '刘晨', '女', 19, 'CS'),
('200215123', '王敏', '女', 18, 'MA'),
('200215124', '陈立', '男', 21, 'CS'),
('200215125', '张立', '男', 19, 'IS'),
('200215126', '赵华', '女', 20, 'CS');

-- 5. 创建 Course 表（课程表）
CREATE TABLE IF NOT EXISTS Course (
    CourseID VARCHAR(10) NOT NULL COMMENT '课程编号',
    CourseName VARCHAR(100) NOT NULL COMMENT '课程名称',
    Credit INT COMMENT '学分',
    PRIMARY KEY (CourseID)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='课程表';

-- 6. 插入课程数据
INSERT INTO Course (CourseID, CourseName, Credit) VALUES
('C001', '数据库系统概论', 4),
('C002', '数据结构', 3),
('C003', '高等数学', 5),
('C004', '计算机网络', 3),
('C005', '操作系统', 4);

-- 7. 创建 SC 表（选课表）- 对应课件中的SC
CREATE TABLE IF NOT EXISTS SC (
    StudentID VARCHAR(20) NOT NULL COMMENT '学号',
    CourseID VARCHAR(10) NOT NULL COMMENT '课程编号',
    Grade INT COMMENT '成绩',
    PRIMARY KEY (StudentID, CourseID),
    FOREIGN KEY (StudentID) REFERENCES Student(ID) ON DELETE CASCADE,
    FOREIGN KEY (CourseID) REFERENCES Course(CourseID) ON DELETE CASCADE
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='选课表';

-- 8. 插入选课数据（包含Grade > 90的成绩，用于课件示例）
INSERT INTO SC (StudentID, CourseID, Grade) VALUES
-- 李勇的选课
('200215121', 'C001', 95),
('200215121', 'C002', 88),
('200215121', 'C003', 92),
-- 刘晨的选课
('200215122', 'C001', 98),
('200215122', 'C004', 85),
-- 王敏的选课
('200215123', 'C002', 78),
('200215123', 'C005', 82),
-- 陈立的选课
('200215124', 'C001', 91),
('200215124', 'C003', 87),
-- 张立的选课
('200215125', 'C001', 76),
('200215125', 'C004', 94),
-- 赵华的选课
('200215126', 'C002', 96),
('200215126', 'C005', 89);

# 嵌套查询
# 另一种等价写法：
$$
B_{F1}(Student \times SC) \text{ as R1}
$$

## F1:
$$
ID = StudentID
$$
$$
B_{Dept} = 'CS' (R1) \text{ as R2}
$$
$$
B_{Grade} > 90 (R2) \text{ as R3}
$$

## II
**ID AS 学号 (R3)**  
- Name AS 姓名  
- CourseID AS 课程号  
- Grade AS 成绩  

---

# select 结果列集 from

select * from  
(  
    select * from  
    (  
        select * from Student, SC where ID = StudentID  
    ) as R1  
    where Dept = 'CS'  
) as R2  
where Grade > 90  

) as R3

# 关系代数表达式

## 步骤1：学生与选课表的笛卡尔积并选择学号相等
$$
R1 \leftarrow \sigma_{Student.ID = SC.StudentID}(Student \times SC)
$$

## 步骤2：选择系别为'CS'的学生
$$
R2 \leftarrow \sigma_{Dept = 'CS'}(R1)
$$

## 步骤3：选择成绩大于90的记录
$$
R3 \leftarrow \sigma_{Grade > 90}(R2)
$$

## 步骤4：投影所需列（学号、姓名、性别、年龄、系别、课程号、成绩）
$$
Result \leftarrow \pi_{ID, Name, Sex, Age, Dept, CourseID, Grade}(R3)
$$

---

# 完整的关系代数表达式

## 写法一：逐步赋值
$$
\begin{align}
R1 &\leftarrow \sigma_{Student.ID = SC.StudentID}(Student \times SC) \\
R2 &\leftarrow \sigma_{Dept = 'CS'}(R1) \\
R3 &\leftarrow \sigma_{Grade > 90}(R2) \\
Result &\leftarrow \pi_{ID,\ Name,\ Sex,\ Age,\ Dept,\ CourseID,\ Grade}(R3)
\end{align}
$$

## 写法二：单行表达式
$$
\pi_{ID,\ Name,\ Sex,\ Age,\ Dept,\ CourseID,\ Grade}(\sigma_{Grade > 90}(\sigma_{Dept = 'CS'}(\sigma_{Student.ID = SC.StudentID}(Student \times SC))))
$$

## 写法三：使用重命名操作（对应SQL中的AS）
$$
\pi_{学号,\ 姓名,\ 性别,\ 年龄,\ 系别,\ 课程号,\ 成绩}(\sigma_{Grade > 90}(\sigma_{Dept = 'CS'}(\sigma_{Student.ID = SC.StudentID}(Student \times SC))))
$$

---

# 关系代数树表示


In [3]:
import pymysql
from tabulate import tabulate

# 数据库连接配置
config = {
    'host': 'localhost',
    'user': 'dylan',
    'password': 'P@ssw0rd',
    'database': 'db_experiment_week03',
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor
}

# 连接数据库
conn = pymysql.connect(**config)
cursor = conn.cursor()

# 执行选择操作：dept = 'CS' 且成绩 > 90 的嵌套查询
cursor.execute("""
SELECT 
    R2.ID AS 学号,
    R2.Name AS 姓名,
    R2.Sex AS 性别,
    R2.Age AS 年龄,
    R2.Dept AS 系别,
    R2.CourseID AS 课程号,
    R2.Grade AS 成绩
FROM (
    SELECT 
        R1.ID, R1.Name, R1.Sex, R1.Age, R1.Dept, 
        R1.CourseID, R1.Grade
    FROM (
        SELECT 
            Student.ID, Student.Name, Student.Sex, 
            Student.Age, Student.Dept, 
            SC.CourseID, SC.Grade
        FROM Student, SC 
        WHERE Student.ID = SC.StudentID
    ) AS R1
    WHERE R1.Dept = 'CS'
) AS R2
WHERE R2.Grade > 90
""")
results = cursor.fetchall()

if results:
    # 打印表格，支持多种格式
    print(tabulate(results, headers="keys", tablefmt="grid"))
    # 其他可选格式： "plain", "simple", "github", "grid", "fancy_grid", "pipe", "orgtbl", "jira", "presto", "pretty"
    
    print(f"\n总记录数: {len(results)} 条")
else:
    print("没有找到符合条件的数据")

cursor.close()
conn.close()

+-----------+--------+--------+--------+--------+----------+--------+
|      学号 | 姓名   | 性别   |   年龄 | 系别   | 课程号   |   成绩 |
+===========+========+========+========+========+==========+========+
| 200215121 | 李勇   | 男     |     20 | CS     | C001     |     95 |
+-----------+--------+--------+--------+--------+----------+--------+
| 200215121 | 李勇   | 男     |     20 | CS     | C003     |     92 |
+-----------+--------+--------+--------+--------+----------+--------+
| 200215122 | 刘晨   | 女     |     19 | CS     | C001     |     98 |
+-----------+--------+--------+--------+--------+----------+--------+
| 200215124 | 陈立   | 男     |     21 | CS     | C001     |     91 |
+-----------+--------+--------+--------+--------+----------+--------+
| 200215126 | 赵华   | 女     |     20 | CS     | C002     |     96 |
+-----------+--------+--------+--------+--------+----------+--------+

总记录数: 5 条


# 连接（[inner] JOIN）运算：笛卡尔积的简化表达，又称为内连接

查询学生的选课信息，结果集包括学生的所有列与选课所有列

## Student 表
| ID | Name | Sex | Age | Dept |
|-----|------|-----|-----|------|
| 200215121 | 李勇 | 男 | 20 | CS |
| 200215122 | 刘晨 | 女 | 19 | CS |
| 200215123 | 王敏 | 女 | 18 | MA |
| 200215125 | 张立 | 男 | 19 | IS |

## SC 表
| StudentID | CourseID | Grade |
|-----------|----------|-------|
| 200215121 | 1 | 92 |
| 200215121 | 2 | 85 |
| 200215121 | 3 | 88 |
| 200215122 | 2 | 90 |
| 200215122 | 3 | 80 |

## 查询过程
从Student表第0行开始（当前行记为t）：对SC表逐行扫描（当前行记为s），如果学号相等 t[ID] = s[StudentID]，把 t ◃▹ s（连接）写到结果表中。扫描完SC的所有行后，t移到下一行，重复上述过程。

## 结果集
| ID | Name | Sex | Age | Dept | StudentID | CourseID | Grade |
|-----|------|-----|-----|------|-----------|----------|-------|
| 200215121 | 李勇 | 男 | 20 | CS | 200215121 | 1 | 92 |
| 200215121 | 李勇 | 男 | 20 | CS | 200215121 | 2 | 85 |
| 200215121 | 李勇 | 男 | 20 | CS | 200215121 | 3 | 88 |
| 200215122 | 刘晨 | 女 | 19 | CS | 200215122 | 2 | 90 |
| 200215122 | 刘晨 | 女 | 19 | CS | 200215122 | 3 | 80 |

-- 内连接（INNER JOIN）查询学生的选课信息

-- 写法1：使用 INNER JOIN 关键字（推荐）
SELECT *
FROM Student
INNER JOIN SC ON Student.ID = SC.StudentID;

-- 写法2：使用 JOIN 关键字（INNER 可省略）
SELECT *
FROM Student
JOIN SC ON Student.ID = SC.StudentID;

-- 写法3：使用 WHERE 子句（隐式内连接）
SELECT *
FROM Student, SC
WHERE Student.ID = SC.StudentID;

-- 写法4：指定需要的列（更规范的写法）
SELECT 
    Student.ID,
    Student.Name,
    Student.Sex,
    Student.Age,
    Student.Dept,
    SC.StudentID,
    SC.CourseID,
    SC.Grade
FROM Student
INNER JOIN SC ON Student.ID = SC.StudentID;

-- 写法5：使用表别名（简化书写）
SELECT 
    s.ID,
    s.Name,
    s.Sex,
    s.Age,
    s.Dept,
    sc.StudentID,
    sc.CourseID,
    sc.Grade
FROM Student s
INNER JOIN SC sc ON s.ID = sc.StudentID;

-- 执行结果：
-- 200215121	李勇	男	20	CS	200215121	1	92
-- 200215121	李勇	男	20	CS	200215121	2	85
-- 200215121	李勇	男	20	CS	200215121	3	88
-- 200215122	刘晨	女	19	CS	200215122	2	90
-- 200215122	刘晨	女	19	CS	200215122	3	80

In [4]:
import pymysql
from tabulate import tabulate

# 数据库连接配置
config = {
    'host': 'localhost',
    'user': 'dylan',
    'password': 'P@ssw0rd',
    'database': 'db_experiment_week03',
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor
}

# 连接数据库
conn = pymysql.connect(**config)
cursor = conn.cursor()

# 执行选择操作：dept = 'CS' 且成绩 > 90 的嵌套查询
cursor.execute("""
SELECT *
FROM Student
WHERE Dept='CS'

UNION

SELECT *
FROM Student
WHERE Dept='IS';
""")
results = cursor.fetchall()

if results:
    # 打印表格，支持多种格式
    print(tabulate(results, headers="keys", tablefmt="grid"))
    # 其他可选格式： "plain", "simple", "github", "grid", "fancy_grid", "pipe", "orgtbl", "jira", "presto", "pretty"
    
    print(f"\n总记录数: {len(results)} 条")
else:
    print("没有找到符合条件的数据")

cursor.close()
conn.close()

+-----------+--------+-------+-------+--------+
|        ID | Name   | Sex   |   Age | Dept   |
+===========+========+=======+=======+========+
| 200215121 | 李勇   | 男    |    20 | CS     |
+-----------+--------+-------+-------+--------+
| 200215122 | 刘晨   | 女    |    19 | CS     |
+-----------+--------+-------+-------+--------+
| 200215124 | 陈立   | 男    |    21 | CS     |
+-----------+--------+-------+-------+--------+
| 200215126 | 赵华   | 女    |    20 | CS     |
+-----------+--------+-------+-------+--------+
| 200215125 | 张立   | 男    |    19 | IS     |
+-----------+--------+-------+-------+--------+

总记录数: 5 条
